# Pattern Detection Visualization
## Visualize what patterns the enhanced model detects

This notebook helps you understand:
- Which Elliott Wave patterns are detected
- Where ranges are identified
- H&S and reversal pattern locations
- Support/resistance level tracking

In [ ]:
import torch
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from environments.simple_trading_env import SimpleTradingEnv
from environments.trading_enhanced_extractor import (
    TradingEnhancedExtractor,
    RangeDetectionCNN,
    ElliottWaveCNN,
    ReversalPatternCNN,
    SupportResistanceCNN
)

print("✓ Loaded pattern detection modules")

In [ ]:
# Load data
DATA_PATH = 'data/binance-BTCUSDT-5m.pkl'
df = pd.read_pickle(DATA_PATH)
test_data = df.iloc[100000:102000].reset_index(drop=True)  # 2000 candles for testing

print(f"Loaded {len(test_data):,} test candles")
print(f"Date range: {test_data['timestamp'].iloc[0]} to {test_data['timestamp'].iloc[-1]}")

In [ ]:
# Create environment and extract features
env = SimpleTradingEnv(test_data, device="cuda", lookback_window=288)
obs, _ = env.reset()

# Extract pattern features
device = torch.device("cuda")

# Convert observations to tensors
ohlc_tensor = torch.from_numpy(obs['price_ohlc_spatial']).unsqueeze(0).float().to(device)

print(f"OHLC tensor shape: {ohlc_tensor.shape}")
print(f"Analyzing patterns...")

In [ ]:
# Initialize pattern detectors
range_detector = RangeDetectionCNN(hidden_dim=32).to(device).eval()
elliott_detector = ElliottWaveCNN(hidden_dim=48).to(device).eval()
reversal_detector = ReversalPatternCNN(hidden_dim=32).to(device).eval()
sr_detector = SupportResistanceCNN(hidden_dim=32).to(device).eval()

# Extract pattern activations
with torch.no_grad():
    range_features = range_detector(ohlc_tensor)
    elliott_features = elliott_detector(ohlc_tensor)
    reversal_features = reversal_detector(ohlc_tensor)
    sr_features = sr_detector(ohlc_tensor)

print(f"✓ Range features: {range_features.shape} - Activation mean: {range_features.mean().item():.3f}")
print(f"✓ Elliott features: {elliott_features.shape} - Activation mean: {elliott_features.mean().item():.3f}")
print(f"✓ Reversal features: {reversal_features.shape} - Activation mean: {reversal_features.mean().item():.3f}")
print(f"✓ S/R features: {sr_features.shape} - Activation mean: {sr_features.mean().item():.3f}")

In [ ]:
# Extract intermediate activations for visualization
# We'll hook into the detectors to get per-timestep activations

activations = {}

def get_activation(name):
    def hook(model, input, output):
        activations[name] = output.detach()
    return hook

# Register hooks
range_detector.volatility_detector.register_forward_hook(get_activation('range_volatility'))
elliott_detector.impulse_fusion.register_forward_hook(get_activation('elliott_impulse'))
elliott_detector.correction_fusion.register_forward_hook(get_activation('elliott_correction'))
reversal_detector.hns_detector.register_forward_hook(get_activation('reversal_hns'))
reversal_detector.double_pattern_detector.register_forward_hook(get_activation('reversal_double'))
reversal_detector.flag_detector.register_forward_hook(get_activation('reversal_flag'))
sr_detector.level_detector.register_forward_hook(get_activation('sr_levels'))

# Run forward pass
with torch.no_grad():
    _ = range_detector(ohlc_tensor)
    _ = elliott_detector(ohlc_tensor)
    _ = reversal_detector(ohlc_tensor)
    _ = sr_detector(ohlc_tensor)

print("✓ Extracted intermediate activations:")
for name, act in activations.items():
    print(f"   {name}: {act.shape}")

In [ ]:
# Visualize patterns on price chart
def plot_patterns_on_price(df_window, activations_dict, start_idx=0, window_size=288):
    """
    Plot price chart with pattern overlays
    """
    # Create subplots
    fig = make_subplots(
        rows=6, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.03,
        subplot_titles=(
            'Price (OHLC)',
            'Range Detection (Volatility)',
            'Elliott Wave - Impulse (12345)',
            'Elliott Wave - Correction (ABC)',
            'Reversal Patterns (H&S, Double Top/Bottom, Flags)',
            'Support/Resistance Levels'
        ),
        row_heights=[0.3, 0.14, 0.14, 0.14, 0.14, 0.14]
    )
    
    # Prepare data
    df_slice = df_window.iloc[start_idx:start_idx+window_size].copy()
    x_axis = np.arange(len(df_slice))
    
    # Row 1: Candlestick chart
    fig.add_trace(
        go.Candlestick(
            x=x_axis,
            open=df_slice['open'],
            high=df_slice['high'],
            low=df_slice['low'],
            close=df_slice['close'],
            name='Price'
        ),
        row=1, col=1
    )
    
    # Row 2: Range detection (volatility compression)
    if 'range_volatility' in activations_dict:
        range_act = activations_dict['range_volatility'][0].cpu().numpy()  # [C, T]
        range_strength = range_act.mean(axis=0)  # Average across channels
        fig.add_trace(
            go.Scatter(
                x=x_axis,
                y=range_strength,
                mode='lines',
                name='Range Strength',
                line=dict(color='purple', width=2),
                fill='tozeroy'
            ),
            row=2, col=1
        )
    
    # Row 3: Elliott Wave - Impulse
    if 'elliott_impulse' in activations_dict:
        impulse_act = activations_dict['elliott_impulse'][0].cpu().numpy()  # [C, T]
        impulse_strength = impulse_act.mean(axis=0)
        fig.add_trace(
            go.Scatter(
                x=x_axis,
                y=impulse_strength,
                mode='lines',
                name='Impulse Wave (12345)',
                line=dict(color='green', width=2),
                fill='tozeroy'
            ),
            row=3, col=1
        )
    
    # Row 4: Elliott Wave - Correction
    if 'elliott_correction' in activations_dict:
        correction_act = activations_dict['elliott_correction'][0].cpu().numpy()  # [C, T]
        correction_strength = correction_act.mean(axis=0)
        fig.add_trace(
            go.Scatter(
                x=x_axis,
                y=correction_strength,
                mode='lines',
                name='Correction Wave (ABC)',
                line=dict(color='red', width=2),
                fill='tozeroy'
            ),
            row=4, col=1
        )
    
    # Row 5: Reversal Patterns
    if 'reversal_hns' in activations_dict:
        hns_act = activations_dict['reversal_hns'][0].cpu().numpy()  # [C, T]
        double_act = activations_dict['reversal_double'][0].cpu().numpy()
        flag_act = activations_dict['reversal_flag'][0].cpu().numpy()
        
        fig.add_trace(
            go.Scatter(
                x=x_axis,
                y=hns_act.mean(axis=0),
                mode='lines',
                name='H&S Pattern',
                line=dict(color='orange', width=1)
            ),
            row=5, col=1
        )
        fig.add_trace(
            go.Scatter(
                x=x_axis,
                y=double_act.mean(axis=0),
                mode='lines',
                name='Double Top/Bottom',
                line=dict(color='blue', width=1)
            ),
            row=5, col=1
        )
        fig.add_trace(
            go.Scatter(
                x=x_axis,
                y=flag_act.mean(axis=0),
                mode='lines',
                name='Flag Pattern',
                line=dict(color='cyan', width=1)
            ),
            row=5, col=1
        )
    
    # Row 6: Support/Resistance
    if 'sr_levels' in activations_dict:
        sr_act = activations_dict['sr_levels'][0].cpu().numpy()  # [C, T]
        sr_strength = sr_act.mean(axis=0)
        fig.add_trace(
            go.Scatter(
                x=x_axis,
                y=sr_strength,
                mode='lines',
                name='S/R Level Strength',
                line=dict(color='brown', width=2),
                fill='tozeroy'
            ),
            row=6, col=1
        )
    
    # Update layout
    fig.update_layout(
        height=1200,
        title_text="Pattern Detection Analysis",
        showlegend=True,
        hovermode='x unified'
    )
    
    fig.update_xaxes(title_text="Candle Index", row=6, col=1)
    
    return fig

# Plot
fig = plot_patterns_on_price(test_data, activations, start_idx=0, window_size=288)
fig.show()

In [ ]:
# Find strongest pattern activations
def find_pattern_peaks(activations_dict, threshold=0.5):
    """
    Find timesteps where patterns are strongly activated
    """
    peaks = {}
    
    for name, act in activations_dict.items():
        if len(act.shape) == 3:  # [B, C, T]
            act_1d = act[0].mean(dim=0).cpu().numpy()  # Average across channels
            
            # Find peaks
            peak_indices = np.where(act_1d > threshold)[0]
            
            if len(peak_indices) > 0:
                peaks[name] = {
                    'indices': peak_indices,
                    'values': act_1d[peak_indices],
                    'max_value': act_1d.max(),
                    'mean_value': act_1d.mean()
                }
    
    return peaks

# Find peaks
peaks = find_pattern_peaks(activations, threshold=0.3)

print("\n📊 PATTERN DETECTION SUMMARY:\n")
for name, info in peaks.items():
    print(f"{name}:")
    print(f"  - Found {len(info['indices'])} strong activations")
    print(f"  - Max activation: {info['max_value']:.3f}")
    print(f"  - Mean activation: {info['mean_value']:.3f}")
    print(f"  - Peak locations: {info['indices'][:5]}...\n")

In [ ]:
# Analyze specific pattern at a peak
def zoom_on_pattern(df, activations_dict, center_idx, window=50):
    """
    Zoom in on a specific pattern detection
    """
    start = max(0, center_idx - window)
    end = min(len(df), center_idx + window)
    
    fig = plot_patterns_on_price(
        df,
        activations_dict,
        start_idx=start,
        window_size=end-start
    )
    
    # Add vertical line at center
    for i in range(1, 7):
        fig.add_vline(
            x=center_idx-start,
            line_dash="dash",
            line_color="red",
            row=i, col=1
        )
    
    fig.update_layout(title_text=f"Pattern Analysis - Center: Candle {center_idx}")
    
    return fig

# Example: Look at first H&S peak
if 'reversal_hns' in peaks:
    first_peak_idx = peaks['reversal_hns']['indices'][0]
    print(f"Zooming on H&S pattern at candle {first_peak_idx}")
    fig = zoom_on_pattern(test_data, activations, first_peak_idx, window=75)
    fig.show()
else:
    print("No H&S patterns detected with current threshold")

## Pattern Interpretation Guide

### Range Detection (Purple)
- **High activation**: Price is consolidating, volatility compressing
- **Look for**: Tight ranges before breakouts
- **Trading signal**: Breakout direction when range resolves

### Elliott Wave - Impulse (Green)
- **High activation**: 5-wave impulse pattern detected
- **Wave structure**: 1-2-3-4-5 (wave 3 is longest)
- **Trading signal**: Enter on wave 3, exit on wave 5 completion

### Elliott Wave - Correction (Red)
- **High activation**: 3-wave correction (ABC) detected
- **Trading signal**: Correction ending = prepare for impulse
- **Bottom/Top**: Wave C completion is the reversal point

### H&S Pattern (Orange)
- **High activation**: Head & Shoulders forming
- **Trading signal**: Neckline break = strong reversal

### Double Top/Bottom (Blue)
- **High activation**: Two tests at same level
- **Trading signal**: Second test failure = reversal

### Flag Pattern (Cyan)
- **High activation**: Continuation pattern (pole + flag)
- **Trading signal**: Breakout continues pole direction

### S/R Levels (Brown)
- **High activation**: Strong support/resistance nearby
- **Trading signal**: Bounce = respect level, Break = breakout